In [6]:
from langchain.document_loaders import PyPDFium2Loader, PyPDFDirectoryLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
import os


In [7]:
# Função para extrair (carregar) arquivos PDF de um diretório
def load_pdf_files(data):
    
    # Cria um loader (carregador) de documentos a partir de um diretório
    loader = DirectoryLoader(
        
        data,  # Caminho da pasta onde estão os arquivos PDF
        
        glob="*.pdf",  # Define que ele deve pegar apenas arquivos com extensão .pdf
        
        loader_cls=PyPDFium2Loader  # Define qual classe será usada para ler os PDFs
                                     # (nesse caso, PyPDFium2Loader, que extrai o texto dos PDFs)
    )

    documents = loader.load()
    return documents

In [8]:
extracted_data = load_pdf_files(
    "C:/Users/kiria/OneDrive/Desktop/PROJETOS/A-Complete-Medical-Chatbot-with-LLMs-LangChain-Pinecone-Flask-AWS/data"
)

In [9]:
len(extracted_data)

4505

In [10]:
from typing import List 
from langchain.schema import Document

#essa função serve para filtrar as informações, o python trouxe várias coisas, como autor do livro, data de publicação e essas informações são irrelevantes.
#então eu quero apenas o source(fonte) e a page content(conteúdo da página)
def filter_to_minimal_docs(docs: list[Document]) -> List[Document]:
    #given a list of document objects, return a new list of document objects containing only 'source' in metadata and the original page_content
    minimal_docs: list[Document] = []


    minimal_docs: list[Document]
    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source":src}
                )
            )
    return minimal_docs

In [11]:
minimal_docs = filter_to_minimal_docs(extracted_data)

In [12]:
# split the document into smaller chunks 
def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20, 
        )
    texts_chunk = text_splitter.split_documents(minimal_docs)
    return texts_chunk

In [13]:
text_chunk = text_split(minimal_docs)
print(f"number of chunks:  {len(text_chunk)}")

number of chunks:  40257


In [14]:
from langchain.embeddings import HuggingFaceEmbeddings

def download_embedding():
    #download and return the Hugging face embeddings model.

    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(
        model_name=model_name,
    )
    return embeddings


embedding = download_embedding()

C:\Users\kiria\AppData\Local\Temp\ipykernel_31196\936082699.py:7: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


In [15]:
embedding

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [16]:
vector = embedding.embed_query("hello word")
print(vector)

#isso aqui é o modelo de embeddings transformando um texto cru em um vetor de números que representa essa frase

[-0.022057322785258293, 0.03135431557893753, 0.04235005006194115, 0.02611093409359455, -0.06519314646720886, -0.07004157453775406, 0.054308995604515076, -0.001612908556126058, -0.06259440630674362, 0.03530619293451309, 0.02730601839721203, 0.051380787044763565, 0.09750931710004807, -0.06328082084655762, 0.008590340614318848, 0.022969596087932587, 0.0637909397482872, -0.026898618787527084, -0.13075026869773865, 0.03945804759860039, -0.0475936159491539, 0.05164392665028572, -0.008419259451329708, 0.000806798052508384, -0.02593027614057064, -0.011951676569879055, -0.015885204076766968, 0.015778236091136932, 0.04706121236085892, -0.07030079513788223, 0.050214849412441254, 0.020775986835360527, 0.13571418821811676, 0.023629747331142426, 0.01985790766775608, 0.035933446139097214, -0.046726327389478683, -0.06226656585931778, -0.013844884000718594, -0.012370181269943714, -0.03176102042198181, -0.09367704391479492, -0.034933846443891525, -0.025743624195456505, 0.07390400022268295, -0.0531868338

In [17]:
print(f"vector lenght: {len(vector)}")

vector lenght: 384


In [46]:
from dotenv import load_dotenv
import os 

load_dotenv()

True

In [19]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

In [20]:
from pinecone import Pinecone
pinecone_api_key = PINECONE_API_KEY

pc = Pinecone(api_key=pinecone_api_key)

In [23]:
from pinecone import ServerlessSpec

index_name= "medical-chatbot"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384, #dimension of embedding
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
        )
    
index = pc.Index(index_name)

#if the vector dimension is high, that means there are more information in the vector

#higher the vector dimension, embedding model will be better

In [24]:
#store our vector -> pinecone

from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=text_chunk,
    embedding=embedding,
    index_name=index_name
    )

In [25]:
#Load existing index

from langchain_pinecone import PineconeVectorStore

#embed each chunk and upsert the embeddings into your pinecone index

docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embedding
    )

# Add more data to the existing Pinecone Index

In [ ]:
kiria = Document(
    page_content="kiria é uma pessoa muito massa é nois",
    metadata={"source": "vozes da minha cabeça"}
)

In [27]:
docsearch.add_documents(documents=[kiria])

['ab9da171-aca9-4918-8d8d-daf0e1b96fe9']

In [28]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3})

#isso aqui significa que voce irá receber 3 respostas relevantes, 3 respostas de maior similaridade da base de conhecimento

In [29]:
retriever_docs = retriever.invoke("what is Acne?")
retriever_docs

[Document(id='41070701-79ca-465f-8d00-d96c0ec5f504', metadata={'source': 'C:\\Users\\kiria\\OneDrive\\Desktop\\PROJETOS\\A-Complete-Medical-Chatbot-with-LLMs-LangChain-Pinecone-Flask-AWS\\data\\The-Gale-Encyclopedia-of-Medicine-3rd-Edition-staibabussalamsula.ac_.id_.pdf'}, page_content='follicles, where pimples form.\nSebum—An oily skin moisturizer produced by\nsebaceous glands.\nTretinoin—A drug that works by increasing the\nturnover (death and replacement) of skin cells.\nAcne vulgaris affecting a woman’s face. Acne is the general\nname given to a skin disorder in which the sebaceous glands\nbecome inflamed. (Photograph by Biophoto Associates, Photo\nResearchers, Inc. Reproduced by permission.)\n26 GALE ENCYCLOPEDIA OF MEDICINE\nAcne'),
 Document(id='72ef3626-3df3-460d-9609-8ef4039943b3', metadata={'source': 'C:\\Users\\kiria\\OneDrive\\Desktop\\PROJETOS\\A-Complete-Medical-Chatbot-with-LLMs-LangChain-Pinecone-Flask-AWS\\data\\The-Gale-Encyclopedia-of-Medicine-3rd-Edition-staibabussa

CONFIGURANDO A LLM
to usando o groq -> api grátis


In [36]:
from langchain_groq import ChatGroq

chatModel = ChatGroq(
    model="llama-3.3-70b-versatile")

In [37]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
system_prompt = (
    "você é um assistente médico para tarefas de perguntas e respostas. "
    "Use as seguintes peças de contexto recuperado para responder "
    "a pergunta. Se você não souber a resposta, diga que você "
    "não sabe. Use três sentenças no máximo e mantenha a resposta conscisa. "
    "\n\n"
    "{context}"
)

#here im creating the system prompt
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)
# a chat model takes 2 inputs, system prompt and user prompt


In [39]:
#create the chain
question_answer_chain = create_stuff_documents_chain(chatModel, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [44]:
response =  rag_chain.invoke({"input": "quem é kiria?"})
print(response["answer"])

Kiria é descrita como uma pessoa muito massa. Não há mais informações disponíveis sobre ela. Não sei mais detalhes sobre quem é Kiria ou o que ela faz.


*AQUI MEU MODELO ESTÁ PRONTO*